In [1]:
# Core Imports and Environment Setup
# Automatically installs missing dependencies if needed

import sys
import os
import subprocess
from pathlib import Path
import time


def setup_environment_and_check_dependencies():
    """
    Set up environment, check dependencies, and verify external tools.
    Handles virtual environment creation, package installation, and tool detection.
    
    Returns:
    - bool: True if setup completed successfully, False otherwise
    """
    # --- Environment Report ---
    divider = "=" * 60
    print(f"{divider}\nEnvironment Check\n{divider}")
    print(f"Python        : {sys.version.split()[0]}")
    print(f"Working dir   : {os.getcwd()}")

    # Check for virtual environment
    venv_exists = Path(".venv").exists()
    if not venv_exists:
        print("⚠ Virtual environment (.venv) not found")
        print("   Creating virtual environment and installing dependencies...")
        try:
            # Run install_dependencies.sh
            result = subprocess.run(
                ["bash", "install_dependencies.sh"],
                capture_output=True,
                text=True,
                timeout=600  # 10 minute timeout
            )
            if result.returncode == 0:
                print("✓ Virtual environment created and dependencies installed")
                print("   Please restart the kernel to use the new environment")
            else:
                print(f"⚠ Installation had issues:\n{result.stderr[:500]}")
                print("   Please run manually: ./install_dependencies.sh")
        except FileNotFoundError:
            print("⚠ install_dependencies.sh not found")
            print("   Creating virtual environment manually...")
            subprocess.run([sys.executable, "-m", "venv", ".venv"], check=False)
        except subprocess.TimeoutExpired:
            print("⚠ Installation timed out")
            print("   Please run manually: ./install_dependencies.sh")
        except Exception as e:
            print(f"⚠ Error during installation: {e}")
            print("   Please run manually: ./install_dependencies.sh")

    # Core package check and auto-install
    missing = []
    for pkg in ["Bio", "numpy", "requests"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print(f"\n⚠ Missing packages: {', '.join(missing)}")
        print("   Attempting to install missing packages...")
        
        # Determine pip command (use venv pip if available)
        pip_cmd = [sys.executable, "-m", "pip"]
        venv_pip = Path(".venv/bin/pip")
        if venv_pip.exists():
            pip_cmd = [str(venv_pip)]
            print("   Using virtual environment pip...")
        
        # Try to install via pip
        try:
            for pkg in missing:
                if pkg == "Bio":
                    pkg_name = "biopython"
                else:
                    pkg_name = pkg
                
                print(f"   Installing {pkg_name}...")
                result = subprocess.run(
                    pip_cmd + ["install", pkg_name, "--quiet"],
                    capture_output=True,
                    text=True,
                    timeout=300
                )
                if result.returncode == 0:
                    print(f"   ✓ {pkg_name} installed")
                else:
                    print(f"   ⚠ Failed to install {pkg_name}")
                    if result.stderr:
                        print(f"   Error: {result.stderr[:200]}")
        except Exception as e:
            print(f"   ⚠ Error installing packages: {e}")
            print("   Please run: ./install_dependencies.sh")
        
        # Re-check after installation attempt
        still_missing = []
        for pkg in missing:
            try:
                __import__(pkg)
            except ImportError:
                still_missing.append(pkg)
        
        if still_missing:
            print(f"\n⚠ Still missing: {', '.join(still_missing)}")
            print("   Please run: ./install_dependencies.sh")
            print("   Or restart kernel after running: ./install_dependencies.sh")
        else:
            print("\n✓ All core packages now available")
    else:
        print("✓ Core packages loaded")

    # Import core packages (now that they're installed)
    # Declare as global to make them available outside the function
    global np, requests, PDB, PDBIO
    try:
        import numpy as np
        import requests
        from Bio import PDB
        from Bio.PDB import PDBIO
        print("✓ Core modules imported")
    except ImportError as e:
        print(f"⚠ Import error: {e}")
        print("   Please restart kernel and run: ./install_dependencies.sh")
        return False

    # Check for external tools
    print("\n" + divider)
    print("External Tools Check")
    print(divider)

    # Check Rosetta
    rosetta_found = False
    rosetta_bin_path = Path("rosetta/source/bin")
    if rosetta_bin_path.exists():
        # Check for Rosetta binaries (they have .linuxgccrelease extension)
        rosetta_binaries = list(rosetta_bin_path.glob("*.linuxgccrelease"))
        if rosetta_binaries:
            # Check for common Rosetta applications
            common_apps = ["relax", "rosetta_scripts", "docking_protocol", "fixbb"]
            found_apps = []
            for app in common_apps:
                if any(app in str(bin_path) for bin_path in rosetta_binaries):
                    found_apps.append(app)
            
            if found_apps:
                print(f"✓ Rosetta (local: rosetta/source/bin)")
                print(f"   Found applications: {', '.join(found_apps)}")
                rosetta_found = True
            else:
                print("✓ Rosetta binaries found (local: rosetta/source/bin)")
                rosetta_found = True
        else:
            print("⚠ Rosetta source found but binaries not built")
            print("   Build with: ./install_rosetta.sh")
    elif Path("rosetta").exists():
        print("⚠ Rosetta directory found but binaries not built")
        print("   Build with: ./install_rosetta.sh")
    else:
        # Check if Rosetta is in PATH
        if subprocess.run(["which", "rosetta_scripts"], capture_output=True).returncode == 0 or \
           subprocess.run(["which", "relax"], capture_output=True).returncode == 0:
            print("✓ Rosetta (in PATH)")
            rosetta_found = True
        else:
            print("⚠ Rosetta not found (optional, used for structure refinement)")
            print("   Install with: ./install_rosetta.sh")

    print(divider)
    return True

# Call the setup function
setup_environment_and_check_dependencies()

Environment Check
Python        : 3.10.12
Working dir   : /home/kuhfeldrf/peptide-md-docking
✓ Core packages loaded
✓ Core modules imported

External Tools Check
✓ Rosetta (local: rosetta/source/bin)
   Found applications: relax, rosetta_scripts, docking_protocol, fixbb


True

In [3]:
# AlphaFold 4 / ColabFold Peptide Structure Prediction
# Predict peptide 3D structure from amino acid sequence using AlphaFold
# Note: All imports are in Cell 0 above

def predict_peptide_structure_alphafold(sequence, output_dir=".", method="colabfold_api"):
    """
    Predict peptide structure using AlphaFold 4 / ColabFold.
    
    Parameters:
    - sequence: Amino acid sequence (single letter code, e.g., "YPFPGP")
    - output_dir: Directory to save output files
    - method: "colabfold_api" (default) or "colabfold_local" or "alphafold_db"
    
    Returns:
    - Path to predicted PDB file, or None if prediction fails
    """
    sequence = sequence.upper().strip()
    
    # Validate sequence
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if not all(aa in valid_aa for aa in sequence):
        invalid = [aa for aa in sequence if aa not in valid_aa]
        print(f"⚠ Invalid amino acids in sequence: {set(invalid)}")
        return None
    
    if len(sequence) < 5:
        print("⚠ Sequence too short (minimum 5 amino acids)")
        return None
    
    if len(sequence) > 2000:
        print("⚠ Sequence too long (maximum 2000 amino acids for peptides)")
        return None
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    output_pdb = output_path / f"{sequence}.pdb"
    
    if method == "colabfold_api":
        return _predict_colabfold_api(sequence, output_pdb)
    elif method == "colabfold_local":
        return _predict_colabfold_local(sequence, output_pdb)
    elif method == "alphafold_db":
        return _predict_alphafold_db(sequence, output_pdb)
    else:
        print(f"⚠ Unknown method: {method}")
        return None


def _predict_colabfold_api(sequence, output_pdb):
    """Predict using ESMFold API (free, fast, requires internet)"""
    try:
        import requests
        
        print(f"🔬 Predicting structure for sequence: {sequence}")
        print(f"   Length: {len(sequence)} amino acids")
        print(f"   Using ESMFold API (Meta AI - fast alternative to AlphaFold)...")
        
        # ESMFold API endpoint (free, fast alternative to AlphaFold)
        # Note: For AlphaFold 4 specifically, use ColabFold locally or AlphaFold Server
        api_url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
        
        print("   Submitting sequence to ESMFold API...")
        response = requests.post(api_url, data=sequence, timeout=120)
        
        if response.status_code == 200:
            # Save PDB file
            with open(output_pdb, 'w') as f:
                f.write(response.text)
            print(f"✓ Structure predicted and saved to: {output_pdb}")
            return str(output_pdb)
        else:
            print(f"⚠ API request failed with status {response.status_code}")
            print(f"   Response: {response.text[:200]}")
            print("   Trying alternative method...")
            return _predict_colabfold_local(sequence, output_pdb)
            
    except ImportError:
        print("⚠ requests library not available. Install with: pip install requests")
        return None
    except Exception as e:
        print(f"⚠ Error calling ESMFold API: {e}")
        print("   Trying alternative method...")
        return _predict_colabfold_local(sequence, output_pdb)


def _predict_colabfold_local(sequence, output_pdb):
    """Predict using local ColabFold installation"""
    try:
        import subprocess
        
        print(f"🔬 Predicting structure using local ColabFold...")
        
        # Check if colabfold_batch is available
        colabfold_cmd = "colabfold_batch"
        result = subprocess.run(["which", colabfold_cmd], capture_output=True)
        
        if result.returncode != 0:
            print("⚠ ColabFold not found locally")
            print("   Install with: pip install colabfold")
            print("   Or use method='colabfold_api' for API-based prediction")
            return None
        
        # Create temporary FASTA file
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as tmp_fasta:
            tmp_fasta.write(f">peptide\n{sequence}\n")
            tmp_fasta_path = tmp_fasta.name
        
        try:
            # Run ColabFold
            cmd = [colabfold_cmd, tmp_fasta_path, str(output_pdb.parent)]
            print(f"   Running: {' '.join(cmd)}")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
            
            if result.returncode == 0:
                # Find the output PDB file
                predicted_files = list(output_pdb.parent.glob(f"*{sequence}*.pdb"))
                if predicted_files:
                    predicted_file = predicted_files[0]
                    if predicted_file != output_pdb:
                        import shutil
                        shutil.copy(predicted_file, output_pdb)
                    print(f"✓ Structure predicted and saved to: {output_pdb}")
                    return str(output_pdb)
                else:
                    print("⚠ Output PDB file not found")
                    return None
            else:
                print(f"⚠ ColabFold failed: {result.stderr}")
                return None
        finally:
            # Clean up temp file
            if os.path.exists(tmp_fasta_path):
                os.unlink(tmp_fasta_path)
                
    except Exception as e:
        print(f"⚠ Error running local ColabFold: {e}")
        return None



def _predict_alphafold_db(sequence, output_pdb):
    """Try to fetch from AlphaFold Database if available"""
    try:
        import requests
        
        print(f"🔬 Searching AlphaFold Database for sequence...")
        
        # AlphaFold Database API
        # Note: This searches for exact matches in the database
        # For custom peptides, use ColabFold instead
        
        # Generate a unique identifier (hash of sequence)
        import hashlib
        seq_hash = hashlib.md5(sequence.encode()).hexdigest()
        
        # Try to fetch from AlphaFold DB (this is a simplified example)
        # In practice, you'd need to use the actual AlphaFold DB API
        print("⚠ AlphaFold Database lookup not fully implemented")
        print("   Use method='colabfold_api' for custom peptide prediction")
        return None
        
    except Exception as e:
        print(f"⚠ Error accessing AlphaFold Database: {e}")
        return None


# predicted_pdb = predict_peptide_structure_alphafold("YPFPGP", method="colabfold_api")
# if predicted_pdb:
#     print(f"Predicted structure saved to: {predicted_pdb}")


# Predict Peptide Structure Using AlphaFold 4

Use AlphaFold 4 / ColabFold to predict 3D structure from peptide sequence


In [ ]:
# Alternative: Predict multiple peptides or batch processing

def predict_multiple_peptides(sequences, output_dir="alphafold_predictions"):
    """
    Predict structures for multiple peptide sequences.
    
    Parameters:
    - sequences: List of amino acid sequences
    - output_dir: Directory to save all predictions
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    results = {}
    
    for i, seq in enumerate(sequences, 1):
        print(f"\n[{i}/{len(sequences)}] Processing: {seq}")
        predicted = predict_peptide_structure_alphafold(
            seq,
            output_dir=output_dir,
            method="colabfold_api"
        )
        results[seq] = predicted
        
        # Add a small delay between requests to avoid rate limiting
        if i < len(sequences):
            time.sleep(2)
    
    print("\n" + "=" * 60)
    print("Batch Prediction Summary")
    print("=" * 60)
    for seq, pdb_file in results.items():
        status = "✓" if pdb_file else "✗"
        print(f"{status} {seq}: {pdb_file or 'Failed'}")
    
    return results

# Helper function to extract peptides from FASTA file
def extract_peptides_from_fasta(fasta_file, num_peptides=5):
    """
    Extract peptide sequences from a FASTA file.
    
    Parameters:
    - fasta_file: Path to FASTA file
    - num_peptides: Number of peptides to extract (default: 5)
    
    Returns:
    - List of peptide sequences
    """
    peptides = []
    try:
        with open(fasta_file, 'r') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('>'):
                    peptides.append(line)
                    if len(peptides) >= num_peptides:
                        break
        print(f"✓ Extracted {len(peptides)} peptides from {fasta_file}")
        return peptides
    except FileNotFoundError:
        print(f"⚠ File not found: {fasta_file}")
        return []
    except Exception as e:
        print(f"⚠ Error reading file: {e}")
        return []

# Example: Extract first 5 peptides from intestinal unique peptides.txt
peptide_file = "intestinal unique peptides.txt"
peptide_sequences = extract_peptides_from_fasta(peptide_file, num_peptides=5)

# Display the extracted peptides
print(f"\nExtracted {len(peptide_sequences)} peptides:")
for i, seq in enumerate(peptide_sequences, 1):
    print(f"  {i}. {seq} (length: {len(seq)} amino acids)")

# Uncomment to run batch prediction:
batch_results = predict_multiple_peptides(peptide_sequences)


✓ Extracted 3 peptides from intestinal unique peptides.txt

Extracted 3 peptides:
  1. GLAPYKLRPVAA (length: 12 amino acids)
  2. LLFKDSAIGF (length: 10 amino acids)
  3. RPKLPLRYP (length: 9 amino acids)

[1/3] Processing: GLAPYKLRPVAA
🔬 Predicting structure for sequence: GLAPYKLRPVAA
   Length: 12 amino acids
   Using ESMFold API (Meta AI - fast alternative to AlphaFold)...
   Submitting sequence to ESMFold API...
✓ Structure predicted and saved to: alphafold_predictions/GLAPYKLRPVAA.pdb

[2/3] Processing: LLFKDSAIGF
🔬 Predicting structure for sequence: LLFKDSAIGF
   Length: 10 amino acids
   Using ESMFold API (Meta AI - fast alternative to AlphaFold)...
   Submitting sequence to ESMFold API...
✓ Structure predicted and saved to: alphafold_predictions/LLFKDSAIGF.pdb

[3/3] Processing: RPKLPLRYP
🔬 Predicting structure for sequence: RPKLPLRYP
   Length: 9 amino acids
   Using ESMFold API (Meta AI - fast alternative to AlphaFold)...
   Submitting sequence to ESMFold API...
